# OilyGiant — Determining the Best Location for New Oil Wells

> **Note:** OilyGiant is a fictional company created exclusively for academic and portfolio purposes. This notebook was developed as part of a Data Science training program and is intended solely to demonstrate technical skills in machine learning, financial analytics, and risk assessment.

---

## 1. Introduction and Objective

You work for the mining company OilyGiant. The task is to find the most profitable and secure location for the development of new oil wells. 

We have access to geological exploration data for three different regions. The objective is to build a machine learning model to predict the volume of reserves in new wells and, subsequently, identify the region that offers the highest total profit. 

**Business Constraints and Project Conditions:**
* **Modeling:** Only Linear Regression must be used to ensure predictability and interpretability.
* **Exploration:** During exploration, a study of 500 points is conducted, from which the top 200 points are selected for profit calculation.
* **Budget:** The budget for developing 200 oil wells is \$100,000,000 USD.
* **Revenue:** One barrel of crude oil yields \$4.50 USD in revenue. Since the volume is measured in thousands of barrels, the revenue per unit is \$4,500 USD.
* **Risk Tolerance:** After assessing risks using the Bootstrapping technique, only regions with a **risk of loss lower than 2.5%** will be considered. The region with the highest average profit among the approved ones will be selected.

The pipeline follows these key steps:
1. Data loading and preparation
2. Model training and validation
3. Financial break-even analysis
4. Risk and profit assessment via Bootstrapping
5. Final region recommendation

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

df0 = pd.read_csv("../data/geo_data_0.csv")
df1 = pd.read_csv("../data/geo_data_1.csv")
df2 = pd.read_csv("../data/geo_data_2.csv")

# Quick inspection across all regions
print("--- Region 0 ---")
display(df0.head())
print(df0.info())
print("Duplicates in Region 0: ", df0.duplicated().sum())
print("\n" + "-"*38 + "\n")

print("--- Region 1 ---")
display(df1.head())
print(df1.info())
print("Duplicates in Region 1: ", df1.duplicated().sum())
print("\n" + "-"*38 + "\n")

print("--- Region 2 ---")
display(df2.head())
print(df2.info())
print("Duplicates in Region 2: ", df2.duplicated().sum())

--- Region 0 ---


,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
None
Duplicates in Region 0:  0

--------------------------------------

--- Region 1 ---


,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305


<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
None
Duplicates in Region 1:  0

--------------------------------------

--- Region 2 ---


,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746


<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
None
Duplicates in Region 2:  0


## 2. Data Preparation

The datasets were loaded successfully with no missing values or duplicated rows. Each dataset contains three geological features (`f0`, `f1`, `f2`) and the target variable (`product` — volume of reserves).

The `id` column is a unique string identifier for each well and carries no predictive weight. It will be removed from all DataFrames to prevent it from interfering with the regression model.

In [2]:
df0 = df0.drop(['id'], axis=1)
df1 = df1.drop(['id'], axis=1)
df2 = df2.drop(['id'], axis=1)

## 3. Model Training and Validation

To avoid code duplication, a reusable function named `train_and_evaluate` is defined below. This function:
1. Splits the data into training (75%) and validation (25%) sets.
2. Trains a `LinearRegression` model.
3. Generates predictions on the validation set.
4. Calculates the average predicted volume and the Root Mean Squared Error (RMSE).

The function returns the true targets and predictions for later use in profit calculations.

In [3]:
def train_and_evaluate(df, region_name):
    X = df.drop(['product'], axis=1)
    y = df['product']

    # 75:25 train-validation split
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=54321)

    # Model training
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predictions
    predictions = model.predict(X_valid)
    predictions = pd.Series(predictions, index=y_valid.index)

    # Metrics
    mean_predicted_volume = predictions.mean()
    rmse = mean_squared_error(y_valid, predictions) ** 0.5

    print(f"--- {region_name} ---")
    print(f"Average predicted volume: {mean_predicted_volume:.2f} thousand barrels")
    print(f"RMSE: {rmse:.2f} thousand barrels\n")

    return y_valid, predictions

target_0, pred_0 = train_and_evaluate(df0, "Region 0")
target_1, pred_1 = train_and_evaluate(df1, "Region 1")
target_2, pred_2 = train_and_evaluate(df2, "Region 2")

--- Region 0 ---
Average predicted volume: 92.16 thousand barrels
RMSE: 37.68 thousand barrels

--- Region 1 ---
Average predicted volume: 68.45 thousand barrels
RMSE: 0.89 thousand barrels

--- Region 2 ---
Average predicted volume: 94.92 thousand barrels
RMSE: 40.15 thousand barrels



**Initial Findings:**
- **Region 0** and **Region 2** show high average predicted volumes (over 90k barrels), but their RMSE values are extremely high (~37k to 40k barrels). This indicates significant variance and poor model precision in these areas.
- **Region 1** has a lower average volume (~68k barrels) but an exceptionally low RMSE (~0.89k barrels). This means the model can predict the volume of reserves in Region 1 with near-perfect accuracy.

## 4. Financial Preparation: Break-Even Analysis

Before selecting wells, we must store the business constraints provided by OilyGiant and determine the break-even point — the minimum volume of barrels a single well must produce to cover its own extraction costs.

In [4]:
# Business Constants
TOTAL_BUDGET_USD = 100000000
TOTAL_WELLS_CHOSEN = 200
REVENUE_PER_UNIT = 4500    # Revenue per 1,000 barrels
RISK_THRESHOLD = 0.025     # Maximum acceptable risk (2.5%)

# Cost and Break-even calculations
cost_per_well = TOTAL_BUDGET_USD / TOTAL_WELLS_CHOSEN
min_volume_no_loss = cost_per_well / REVENUE_PER_UNIT

print(f"Total Budget for {TOTAL_WELLS_CHOSEN} wells: USD {TOTAL_BUDGET_USD:,.2f}")
print(f"Cost per well: USD {cost_per_well:,.2f}")
print(f"Minimum volume required per well (Break-even): {min_volume_no_loss:.2f} thousand barrels")

Total Budget for 200 wells: USD 100,000,000.00
Cost per well: USD 500,000.00
Minimum volume required per well (Break-even): 111.11 thousand barrels


The break-even volume is **111.11 thousand barrels** per well. 

Comparing this threshold to the regional averages (~68k to ~94k), it is evident that random drilling would lead to massive financial losses. The Machine Learning model is essential here: it must identify the specific, highly concentrated points within a region that exceed the 111.11k mark.

## 5. Profit Calculation Function

The function below calculates the potential profit for a given set of wells. Following the business rules, we select the top 200 wells based on the model's **predictions**, but the financial return is calculated using the **true volume** (target) of those selected wells.

In [5]:
def calculate_profit(target, predictions, count):
    # Sort predictions in descending order to find the best wells
    pred_sorted = predictions.sort_values(ascending=False)
    
    # Select the true targets corresponding to the top predictions
    selected_targets = target[pred_sorted.index][:count] 
    total_volume = selected_targets.sum() 

    revenue = total_volume * REVENUE_PER_UNIT
    profit = revenue - TOTAL_BUDGET_USD
    return profit

print(f"Ideal Profit - Region 0: USD {calculate_profit(target_0, pred_0, TOTAL_WELLS_CHOSEN):,.2f}")
print(f"Ideal Profit - Region 1: USD {calculate_profit(target_1, pred_1, TOTAL_WELLS_CHOSEN):,.2f}")
print(f"Ideal Profit - Region 2: USD {calculate_profit(target_2, pred_2, TOTAL_WELLS_CHOSEN):,.2f}")

Ideal Profit - Region 0: USD 31,786,315.96
Ideal Profit - Region 1: USD 24,150,866.97
Ideal Profit - Region 2: USD 24,137,856.36


The values above represent an ideal, theoretical scenario where we select the absolute best 200 wells from the entire validation set. However, OilyGiant's real-world exploration strategy involves analyzing samples of only **500 points** at a time. 

To simulate this reality and accurately evaluate the risk of financial loss, we must apply the Bootstrapping technique.

## 6. Risk and Profit Assessment via Bootstrapping

Bootstrapping is applied using 1,000 subsamples to simulate the exploration process. In each iteration:
1. We randomly sample 500 wells from the region.
2. We use the model to pick the top 200 wells out of those 500.
3. We calculate the profit and store it in a distribution.

Finally, we compute the average expected profit, the 95% confidence interval, and the probability of a negative profit (Risk of Loss).

In [6]:
state = np.random.RandomState(54321)

def evaluate_risk(target, predictions, region_name):
    values = []

    # Bootstrapping with 1,000 iterations
    for i in range(1000):
        target_subsample = target.sample(n=500, replace=True, random_state=state)
        pred_subsample = predictions[target_subsample.index]

        profit = calculate_profit(target_subsample, pred_subsample, TOTAL_WELLS_CHOSEN)
        values.append(profit)

    values = pd.Series(values)

    mean_profit = values.mean()
    lower_quantile = values.quantile(0.025)
    upper_quantile = values.quantile(0.975)

    risk_of_loss = (values < 0).mean() * 100

    print(f"--- Bootstrapping Results: {region_name} ---")
    print(f"Expected Average Profit: USD {mean_profit:,.2f}")
    print(f"95% Confidence Interval: USD {lower_quantile:,.2f} to USD {upper_quantile:,.2f}")
    print(f"Risk of Loss: {risk_of_loss:.2f}%")

    if risk_of_loss < (RISK_THRESHOLD * 100):
        print("STATUS: APPROVED (Risk < 2.5%)\n")
    else:
        print("STATUS: REJECTED (Risk > 2.5%)\n")

evaluate_risk(target_0, pred_0, "Region 0")
evaluate_risk(target_1, pred_1, "Region 1")
evaluate_risk(target_2, pred_2, "Region 2")

--- Bootstrapping Results: Region 0 ---
Expected Average Profit: USD 4,089,561.93
95% Confidence Interval: USD -963,570.86 to USD 9,616,291.73
Risk of Loss: 5.20%
STATUS: REJECTED (Risk > 2.5%)

--- Bootstrapping Results: Region 1 ---
Expected Average Profit: USD 4,739,087.11
95% Confidence Interval: USD 738,849.77 to USD 9,479,263.62
Risk of Loss: 1.20%
STATUS: APPROVED (Risk < 2.5%)

--- Bootstrapping Results: Region 2 ---
Expected Average Profit: USD 4,280,025.26
95% Confidence Interval: USD -1,129,684.21 to USD 9,985,575.18
Risk of Loss: 6.10%
STATUS: REJECTED (Risk > 2.5%)



## 7. Conclusion

The primary goal of this project was to identify the safest and most profitable region for OilyGiant's new wells, strictly adhering to a maximum risk tolerance of 2.5%.

| Region | Expected Profit (USD) | Risk of Loss | Status |
|---|---|---|---|
| Region 0 | ~$4.08M | 5.20% | ❌ Rejected |
| **Region 1** | **~$4.73M** | **1.20%** | **✅ Approved** |
| Region 2 | ~$4.28M | 6.10% | ❌ Rejected |

**Final Recommendation:**
**Region 1 is unequivocally the best location for development.**

Although Regions 0 and 2 seemed promising due to their higher initial volume averages, the initial modeling phase revealed massive prediction errors (RMSE > 37k). Region 1, conversely, allowed the linear regression model to make near-perfect predictions (RMSE < 1k). 

This extreme predictability directly mitigated financial risk during the Bootstrapping phase. Region 1 is the only location that satisfies OilyGiant's strict risk criteria (1.2% risk vs. the 2.5% limit) and provides a highly secure lower bound in the 95% confidence interval, ensuring profitability even in less-than-ideal sampling scenarios.